# Numerical Solution of the Blasius Boundary - Value Problem
## Anchal Bhupal
## SC26M006

## 1. Aim
This assignment aims to obtain a numerical solution of the Blasius equation for laminar flow over a flat plate. The boundary-value problem is solved by converting it into an Initial-Value Problem using the shooting method. We implement third- and fourth-order Runge-Kutta schemes and examine their accuracy for different step sizes.

## 2. Governing equation and Boundary conditions
The similarity form of the Blasius equation is:
$$f''' + \frac{1}{2}ff'' = 0, \, 0 \leq \eta < \infty.$$
The boundary conditions are:
$$f(0)=0, f'(0)=0, f'(\infty)=1.$$
The first two conditions represent no penetration and no slip at the plate. The last condition requires the velocity to approach the free-stream value away from the wall. In the numerical calculation, infinity is represented by a sufficiently large finite value, chosen here as $\eta_{max}=10$.

## 3. Conversion into a first-order system
Introduce three dependent variables:
$$Y_1 = f, Y_2 = f', Y_3 = f'' .$$
Differentiating these definitions and using the governing equation gives:
$$Y'_1 = Y_2, Y'_2 = Y_3, Y'_3 = - \frac{1}{2}Y_1 Y_3$$
Therefore,
$$\frac{d \mathbf{Y}}{d\eta} = \mathbf{G(Y)} = \begin{bmatrix}Y_2\\Y_3\\-\frac{1}{2}Y_1 Y_3 \end{bmatrix}$$
At the wall, $Y_1(0)=0$ and $Y_2(0)=0$ are known, but $Y_3(0)=f''(0)$ is not specified. We denote this unknown initial curvature by $s$, so the initial state becomes $\mathbf{Y}(0)=[0,0,s]^T$

## 4. Shooting method 
A trial value of $s$ makes the problem an initial value problem. The system is integrated from $\eta = 0$ to $\eta = 10$, and the computed terminal slope is compared with the required value of one. We define:
$$R(s) = f'(10:s)-1$$
The required value of $s$ is the root of $R(s)=0$. Bisection is used because the residual changes sign over the initial bracket $[0.2, 0.4]$.\\
The midpoint is evaluated repeatedly; the half interval containing the sign change is retained until the interval width or residual is sufficiently small.

## 5. Numerical integration
### 5.1 Third-order Runge-Kutta method
For a system $Y' = G(Y)$, the third-order scheme used is: 
$$K_1 = hG(Y_n),$$
$$K_2 = hG(Y_n + K_1/2),$$
$$K_3 = hG(Y_n - K_1 +2K_2),$$
$$Y_{n+1} = Y_n + \frac{K_1 + 4K_2 + K_3}{6} .$$

### 5.2 Fourth-order Runge-Kutta method
The typical fourth-order scheme is:
$$K_1 = hG(Y_N),$$
$$K_2 = hG(Y_n + K_1/2),$$
$$K_3 = hG(Y_n - K_1 +2K_2),$$
$$Y_{n+1} = Y_n + \frac{K_1 + 2K_2 + 2K-3 + K_4}{6} .$$

The expected global errors are proportional to $h^3$ for RK3 and $h^4$ for RK4, before round-off and other error sources become significant.

In [2]:
#Defining functions and parameters

import numpy as np
import matplotlib.pyplot as plt

eta_max = 10.0
step_size = [0.4, 0.2, 0.1, 0.05, 0.025]
reference_s =0.3321

def G(Y):
    f ,fp, fpp =Y
    return np.array([fp, fpp, -0.5*fp*fpp], dtype=float)

def integrate_ivp(s, h, method, save_history=False):
    n=int(round(eta_max/h))
    h= eta_max/n
    Y=np.array([0.0,0.0,s])
    history = [Y.copy()]

    for i in range(n):
        K1 = h*G(Y)
        if method=="RK3":
            K2 = h*G(Y+K1/2)
            K3 = h*G(Y- K1 + 2*K2)
            Y=Y+(K1+4*K2+K3)/6
        elif method == "RK4":
            K2=h*G(Y+K1/2)
            K3=h*G(Y+K2/2)
            K4=h*G(Y + K3)
            Y=Y+(K1+2*K2+2*K3+K4)/6
        else:
            raise ValueError("Choose RK3 or RK4.")
            history.append(Y.copy())

        if save_history:
            eta = np.linspace(0.0, eta_max, n+1)
            return eta, np.asarray(history)
        return Y

def terminal_residual(s, h, method):
    return integrate_ivp(s,h, method)[1]-1.0

def find_shotting_parameter(h,  method, lower=0.2, upper=0.4, tol=1e-14):
    a, b = lower, upper
    fa = terminal_residual(a, h, method)
    fb = terminal_residual(b,h , method)
    if fa*fb>0:
        raise ValueError("The initial interval does not bracket a root.")

    for i in range(100):
        mid = 0.5*(a+b)
        fm = terminal_residual(mid, h, method)

        if abs(fm)<tol or 0.5*(b-a) < tol:
            return mid
        if fa*fm<0:
            b, fb = mid, fm
        else:
            a, fa = mid, fm
    return 0.5*(a+b)

## 6. Third-order Runge-Kutta solution and results